In [ ]:
# Environment Configuration & Setup
ENVIRONMENT = "kaggle"
# ENVIRONMENT = "local"

if ENVIRONMENT == "kaggle":
    TRAIN_PATH = "/kaggle/input/datasets/deanfebrio/aol-textmining-fakenewsdetection-dataset/processed/train_processed.csv"
    TEST_PATH = "/kaggle/input/datasets/deanfebrio/aol-textmining-fakenewsdetection-dataset/processed/test_processed.csv"
    OUTPUT_DIR = "/kaggle/working/model/"
elif ENVIRONMENT == "local":
    TRAIN_PATH = "./data/processed/train_processed.csv"
    TEST_PATH = "./data/processed/test_processed.csv"
    OUTPUT_DIR = "./artifacts/model/"
else:
    raise ValueError(f"Unknown environment: {ENVIRONMENT}")

print(f"Environment: {ENVIRONMENT}")
print(f"Train path: {TRAIN_PATH}")
print(f"Test path: {TEST_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
if ENVIRONMENT == "kaggle":
  !pip install -q pandas numpy matplotlib seaborn scikit-learn torch transformers datasets accelerate tqdm

# Setup

Import library, konfigurasi random seed, dan deteksi device.

In [ ]:
import pandas as pd
import numpy as np
import os

import torch
import transformers

# ==========================================
# MONKEY PATCH FOR DATAPARALLEL BUG
# ==========================================
# Patch PreTrainedModel.dtype to prevent StopIteration in DataParallel
@property
def patched_dtype(self):
    try:
        return next(param.dtype for param in self.parameters() if param.is_floating_point())
    except StopIteration:
        return torch.float32
transformers.modeling_utils.PreTrainedModel.dtype = patched_dtype
# ==========================================
from transformers import (
    LongformerTokenizerFast,
    LongformerForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import json
import warnings

warnings.filterwarnings("ignore")

# Random seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device detection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Torch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Device: {device}")

# Load Dataset

Memuat dataset training dan testing yang sudah dipreprocessing.

In [ ]:
df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

print("=== Train Dataset ===")
print(f"Shape: {df_train.shape}")
print(f"Columns: {df_train.columns.tolist()}")

print("\n=== Test Dataset ===")
print(f"Shape: {df_test.shape}")
print(f"Columns: {df_test.columns.tolist()}")

# Verify required columns
for col in ["Headline", "articleBody", "Stance"]:
    assert col in df_train.columns, f"Missing column in train: {col}"
    print(f"Column '{col}' verified in train dataset")

for col in ["Headline", "articleBody"]:
    assert col in df_test.columns, f"Missing column in test: {col}"
    print(f"Column '{col}' verified in test dataset")

print("\n=== Label Distribution (Train) ===")
print(df_train["Stance"].value_counts())

# Label Encoding

Mapping label stance ke nilai numerik untuk training.

In [ ]:
label2id = {
    "agree": 0,
    "disagree": 1,
    "discuss": 2,
    "unrelated": 3,
}

id2label = {v: k for k, v in label2id.items()}

print("label2id:", label2id)
print("id2label:", id2label)

# Convert labels to numeric
df_train["label"] = df_train["Stance"].map(label2id)

print("\n=== Label Counts ===")
print(df_train["label"].value_counts().sort_index())

# Verify no NaN labels
assert df_train["label"].isna().sum() == 0, "Found unmapped labels!"
print("\nAll labels mapped successfully.")

# Train Validation Split

Stratified split 90% train dan 10% validation untuk menjaga distribusi kelas.

In [ ]:
df_train_split, df_val_split = train_test_split(
    df_train,
    test_size=0.1,
    random_state=SEED,
    stratify=df_train["label"],
)

df_train_split = df_train_split.reset_index(drop=True)
df_val_split = df_val_split.reset_index(drop=True)

print(f"Train split size: {len(df_train_split)}")
print(f"Validation split size: {len(df_val_split)}")

print("\n=== Train Split Distribution ===")
print(df_train_split["label"].value_counts().sort_index())

print("\n=== Validation Split Distribution ===")
print(df_val_split["label"].value_counts().sort_index())

# Longformer Initialization

Load tokenizer dan model dari `allenai/longformer-base-4096` dengan konfigurasi 4 label untuk multi-class classification.

In [ ]:
MODEL_NAME = "allenai/longformer-base-4096"

tokenizer = LongformerTokenizerFast.from_pretrained(MODEL_NAME)

model = LongformerForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=label2id,
    id2label=id2label,
)

model.to(device)

print(f"Model: {MODEL_NAME}")
print(f"Num labels: {model.config.num_labels}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Token Length Analysis

Analisis panjang token sebelum menentukan MAX_LENGTH untuk tokenisasi. Menggunakan tokenizer Longformer untuk menghitung jumlah token pada headline, article body, dan gabungan keduanya.

In [ ]:
def count_tokens(text):
    """Count tokens for a given text using the loaded tokenizer."""
    return len(tokenizer.encode(str(text), add_special_tokens=False))

# Sample for analysis (use full train split)
print("Calculating token lengths (this may take a moment)...")

headline_tokens = df_train_split["Headline"].apply(count_tokens)
article_tokens = df_train_split["articleBody"].apply(count_tokens)
# Combined: headline + sep + article + special tokens (~4 tokens overhead)
combined_tokens = headline_tokens + article_tokens + 4

def report_token_stats(series, name):
    print(f"\n--- {name} ---")
    print(f"Mean:   {series.mean():.1f}")
    print(f"Median: {series.median():.1f}")
    print(f"Max:    {series.max()}")
    print(f"P95:    {np.percentile(series, 95):.1f}")
    print(f"P99:    {np.percentile(series, 99):.1f}")

report_token_stats(headline_tokens, "Headline Token Length")
report_token_stats(article_tokens, "Article Token Length")
report_token_stats(combined_tokens, "Combined Token Length")

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(headline_tokens, bins=30, kde=True, ax=axes[0, 0])
axes[0, 0].set_title("Headline Token Length Distribution")

sns.histplot(article_tokens, bins=50, kde=True, ax=axes[0, 1])
axes[0, 1].set_title("Article Token Length Distribution")

sns.histplot(combined_tokens, bins=50, kde=True, ax=axes[1, 0])
axes[1, 0].set_title("Combined Token Length Distribution")

sns.boxplot(data=pd.DataFrame({
    "Headline": headline_tokens,
    "Article": article_tokens,
    "Combined": combined_tokens,
}), ax=axes[1, 1])
axes[1, 1].set_title("Token Length Comparison")

plt.tight_layout()
plt.show()

# Dynamic Max Length Selection

Nilai `MAX_LENGTH` menentukan panjang maksimum token yang akan diproses oleh model. Nilai ini sebaiknya disesuaikan berdasarkan hasil analisis token length di atas.

Pertimbangan:
- Nilai yang terlalu besar membutuhkan lebih banyak GPU memory dan waktu training
- Nilai yang terlalu kecil akan memotong (truncate) banyak artikel
- Longformer mendukung hingga 4096 token
- Pilih nilai yang mencakup mayoritas data (misal P95 atau P99)

In [ ]:
# Adjust this value based on Token Length Analysis results
MAX_LENGTH = 1024

print(f"MAX_LENGTH: {MAX_LENGTH}")
print(f"Percentage of combined tokens within MAX_LENGTH: "
      f"{(combined_tokens <= MAX_LENGTH).mean() * 100:.2f}%")

# Global Attention Configuration

Longformer menggunakan kombinasi local attention dan global attention. Global attention memungkinkan token tertentu untuk attend ke semua token lainnya.

Mode yang didukung:
- `cls_only`: Hanya token CLS yang mendapat global attention
- `headline_and_cls`: Token CLS dan semua token headline mendapat global attention (direkomendasikan untuk stance detection karena headline adalah referensi utama)

In [ ]:
# Global attention mode: "cls_only" or "headline_and_cls"
GLOBAL_ATTENTION_MODE = "headline_and_cls"

print(f"Global attention mode: {GLOBAL_ATTENTION_MODE}")

# Tokenization

Tokenisasi headline dan article body menggunakan Longformer tokenizer. Headline dan article body digabungkan sebagai input pair (text, text_pair) sehingga tokenizer otomatis menambahkan separator token.

In [ ]:
def tokenize_and_prepare(df, has_labels=True):
    """
    Tokenize dataframe and prepare global attention masks.

    Args:
        df: DataFrame with 'Headline' and 'articleBody' columns
        has_labels: Whether the dataframe has 'label' column

    Returns:
        dict with input_ids, attention_mask, global_attention_mask, and optionally labels
    """
    # Tokenize headline + article as text pair
    encodings = tokenizer(
        df["Headline"].astype(str).tolist(),
        df["articleBody"].astype(str).tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors=None,
    )

    # Generate global attention masks
    global_attention_masks = []
    for i, input_ids in enumerate(encodings["input_ids"]):
        global_attn = [0] * len(input_ids)

        if GLOBAL_ATTENTION_MODE == "cls_only":
            # Only CLS token gets global attention
            global_attn[0] = 1

        elif GLOBAL_ATTENTION_MODE == "headline_and_cls":
            # CLS token gets global attention
            global_attn[0] = 1
            # All headline tokens get global attention (tokens before first </s>)
            for j, token_id in enumerate(input_ids):
                if j == 0:
                    continue
                if token_id == tokenizer.sep_token_id:
                    break
                global_attn[j] = 1
        else:
            raise ValueError(f"Unknown global attention mode: {GLOBAL_ATTENTION_MODE}")

        global_attention_masks.append(global_attn)

    result = {
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "global_attention_mask": global_attention_masks,
    }

    if has_labels:
        result["labels"] = df["label"].tolist()

    return result

print("Tokenizing train split...")
train_encodings = tokenize_and_prepare(df_train_split, has_labels=True)

print("Tokenizing validation split...")
val_encodings = tokenize_and_prepare(df_val_split, has_labels=True)

print("Tokenizing test set...")
test_encodings = tokenize_and_prepare(df_test, has_labels=False)

print("\nTokenization complete.")
print(f"Train samples: {len(train_encodings['input_ids'])}")
print(f"Validation samples: {len(val_encodings['input_ids'])}")
print(f"Test samples: {len(test_encodings['input_ids'])}")

# Hugging Face Dataset

Konversi hasil tokenisasi ke format Hugging Face Dataset dan set format PyTorch.

In [ ]:
train_dataset = Dataset.from_dict(train_encodings)
val_dataset = Dataset.from_dict(val_encodings)
test_dataset = Dataset.from_dict(test_encodings)

# Set PyTorch format
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "global_attention_mask", "labels"],
)
val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "global_attention_mask", "labels"],
)
test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "global_attention_mask"],
)

print(f"Train dataset: {train_dataset}")
print(f"Validation dataset: {val_dataset}")
print(f"Test dataset: {test_dataset}")

# Class Weight Analysis

Menghitung bobot kelas berdasarkan distribusi label di training set. Bobot ini dapat digunakan untuk menangani class imbalance.

In [ ]:
train_labels = np.array(df_train_split["label"].tolist())

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels,
)

class_weight_dict = {i: w for i, w in enumerate(class_weights)}

print("=== Class Frequencies ===")
for label_id in sorted(id2label.keys()):
    count = (train_labels == label_id).sum()
    pct = count / len(train_labels) * 100
    print(f"  {id2label[label_id]:>10s} (id={label_id}): {count:>6d} ({pct:.2f}%)")

print("\n=== Class Weights ===")
for label_id in sorted(id2label.keys()):
    print(f"  {id2label[label_id]:>10s} (id={label_id}): {class_weights[label_id]:.4f}")

# Store as tensor for potential custom loss usage
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"\nClass weights tensor: {class_weights_tensor}")

# Metrics Function

Fungsi `compute_metrics()` untuk evaluasi model selama training. Menghitung accuracy, precision, recall, F1 macro, dan F1 weighted.

In [ ]:
def compute_metrics(eval_pred):
    """
    Compute evaluation metrics for the Trainer.

    Args:
        eval_pred: EvalPrediction object with predictions and label_ids

    Returns:
        dict with accuracy, precision_macro, recall_macro, f1_macro, f1_weighted
    """
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
    }

print("compute_metrics() defined.")

# Training Configuration

Konfigurasi `TrainingArguments` untuk Hugging Face Trainer. Parameter dapat disesuaikan berdasarkan GPU yang tersedia dan kebutuhan training.

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    num_train_epochs=3,
    fp16=True,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    seed=SEED,
    report_to="none",
    dataloader_num_workers=0,
    remove_unused_columns=False,
)

print("=== Training Configuration ===")
print(f"Batch size (per device): {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Weight decay: {training_args.weight_decay}")
print(f"Warmup ratio: {training_args.warmup_ratio}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"FP16: {training_args.fp16}")
print(f"Best model metric: {training_args.metric_for_best_model}")

# Early Stopping

Callback untuk menghentikan training lebih awal jika metrik evaluasi tidak membaik setelah beberapa epoch.

In [ ]:
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.0,
)

print(f"Early stopping patience: {early_stopping.early_stopping_patience}")

# Trainer Initialization

Inisialisasi Hugging Face Trainer dengan model, tokenizer, dataset, metrics, dan callbacks.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping],
)

print("Trainer initialized successfully.")
print(f"Train dataset size: {len(train_dataset)}")
print(f"Eval dataset size: {len(val_dataset)}")

# Training

Eksekusi training model Longformer. Proses ini akan:
1. Training selama beberapa epoch sesuai konfigurasi
2. Evaluasi di setiap akhir epoch
3. Menyimpan checkpoint terbaik berdasarkan F1 macro
4. Menghentikan training lebih awal jika tidak ada peningkatan (early stopping)

In [ ]:
start_time = time.time()


import os
from transformers.trainer_utils import get_last_checkpoint

# Cek apakah ada checkpoint sebelumnya di folder output
last_checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

if last_checkpoint is not None:
    print(f"Melanjutkan training dari checkpoint: {last_checkpoint}")
    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Memulai training dari awal...")
    train_result = trainer.train()

training_time = time.time() - start_time
print(f"\nTraining completed in {training_time / 60:.2f} minutes")
print(f"\n=== Training Results ===")
for key, value in train_result.metrics.items():
    print(f"  {key}: {value}")

# Validation Evaluation

Evaluasi model pada validation set untuk melihat performa sebelum evaluasi pada test set.

In [ ]:
print("Evaluating on validation set...")

val_predictions = trainer.predict(val_dataset)
val_preds = np.argmax(val_predictions.predictions, axis=-1)
val_labels = np.array(df_val_split["label"].tolist())

# Classification report
target_names = [id2label[i] for i in range(4)]
print("\n=== Validation Classification Report ===")
print(classification_report(val_labels, val_preds, target_names=target_names, digits=4))

# Key metrics
print(f"F1 Macro:    {f1_score(val_labels, val_preds, average='macro'):.4f}")
print(f"F1 Weighted: {f1_score(val_labels, val_preds, average='weighted'):.4f}")

# Confusion matrix
val_cm = confusion_matrix(val_labels, val_preds)
print("\n=== Validation Confusion Matrix ===")
print(val_cm)

# Test Evaluation

Evaluasi model pada test set. Catatan: test set FNC-1 yang tersedia mungkin tidak memiliki label (unlabeled). Jika kolom Stance tersedia di test set, evaluasi akan dilakukan. Jika tidak, hanya prediksi yang dihasilkan.

In [ ]:
print("Evaluating on test set...")

test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=-1)
test_pred_labels = [id2label[p] for p in test_preds]

# Check if test set has labels
if "Stance" in df_test.columns:
    df_test["label"] = df_test["Stance"].map(label2id)
    test_labels = np.array(df_test["label"].tolist())

    target_names = [id2label[i] for i in range(4)]
    print("\n=== Test Classification Report ===")
    print(classification_report(test_labels, test_preds, target_names=target_names, digits=4))

    print(f"Accuracy:    {accuracy_score(test_labels, test_preds):.4f}")
    print(f"Precision:   {precision_score(test_labels, test_preds, average='macro'):.4f}")
    print(f"Recall:      {recall_score(test_labels, test_preds, average='macro'):.4f}")
    print(f"F1 Macro:    {f1_score(test_labels, test_preds, average='macro'):.4f}")
    print(f"F1 Weighted: {f1_score(test_labels, test_preds, average='weighted'):.4f}")

    test_cm = confusion_matrix(test_labels, test_preds)
    print("\n=== Test Confusion Matrix ===")
    print(test_cm)
else:
    print("Test set does not have Stance labels. Showing prediction distribution only.")
    print(pd.Series(test_pred_labels).value_counts())

# Error Analysis

Menampilkan 20 sampel yang salah diklasifikasi untuk analisis kesalahan model.

In [ ]:
def error_analysis(df, true_labels, pred_labels, id2label, n=20):
    """
    Display top N misclassified samples.

    Args:
        df: Original dataframe
        true_labels: Array of true label ids
        pred_labels: Array of predicted label ids
        id2label: Mapping from id to label name
        n: Number of samples to display
    """
    misclassified = np.where(true_labels != pred_labels)[0]
    print(f"Total misclassified: {len(misclassified)} / {len(true_labels)} "
          f"({len(misclassified) / len(true_labels) * 100:.2f}%)")

    sample_indices = misclassified[:n]
    print(f"\nShowing top {len(sample_indices)} misclassified samples:\n")

    for rank, idx in enumerate(sample_indices, 1):
        headline = str(df.iloc[idx]["Headline"])
        body = str(df.iloc[idx]["articleBody"])[:200]
        true = id2label[true_labels[idx]]
        pred = id2label[pred_labels[idx]]

        print(f"--- Sample {rank} (index {idx}) ---")
        print(f"True: {true:>10s}  |  Predicted: {pred:>10s}")
        print(f"Headline: {headline}")
        print(f"Article:  {body}...")
        print()

# Run error analysis on validation set
error_analysis(df_val_split, val_labels, val_preds, id2label, n=20)

# Confusion Matrix Visualization

Visualisasi confusion matrix menggunakan seaborn heatmap.

In [ ]:
def plot_confusion_matrix(cm, labels, title="Confusion Matrix"):
    """
    Plot confusion matrix as a seaborn heatmap.

    Args:
        cm: Confusion matrix array
        labels: List of label names
        title: Plot title
    """
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
    )
    plt.title(title)
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.show()

target_names = [id2label[i] for i in range(4)]

# Validation confusion matrix
plot_confusion_matrix(val_cm, target_names, title="Validation Confusion Matrix")

# Test confusion matrix (if labels available)
if "Stance" in df_test.columns:
    plot_confusion_matrix(test_cm, target_names, title="Test Confusion Matrix")

# Model Saving

Menyimpan model, tokenizer, dan konfigurasi label ke output directory.

In [ ]:
save_dir = Path(OUTPUT_DIR)
save_dir.mkdir(parents=True, exist_ok=True)

# Save model and tokenizer
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

# Save label mappings
label_config = {
    "label2id": label2id,
    "id2label": {str(k): v for k, v in id2label.items()},
}
with open(save_dir / "label_config.json", "w") as f:
    json.dump(label_config, f, indent=2)

# Save training configuration summary
train_config = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "global_attention_mode": GLOBAL_ATTENTION_MODE,
    "batch_size": training_args.per_device_train_batch_size,
    "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
    "learning_rate": training_args.learning_rate,
    "weight_decay": training_args.weight_decay,
    "warmup_ratio": training_args.warmup_ratio,
    "num_epochs": training_args.num_train_epochs,
    "fp16": training_args.fp16,
    "seed": SEED,
    "environment": ENVIRONMENT,
}
with open(save_dir / "training_config.json", "w") as f:
    json.dump(train_config, f, indent=2)

print(f"Model saved to: {save_dir}")
print(f"Files saved:")
for f_path in sorted(save_dir.iterdir()):
    print(f"  {f_path.name}")

# Inference Example

Fungsi `predict_stance()` untuk prediksi stance pada input baru. Dapat digunakan untuk inferensi setelah model selesai ditraining.

In [ ]:
def predict_stance(headline, article_body, model=model, processing_class=tokenizer):
    """
    Predict stance for a headline-article pair.

    Args:
        headline: News headline text
        article_body: News article body text
        model: Trained LongformerForSequenceClassification
        tokenizer: LongformerTokenizerFast

    Returns:
        tuple: (predicted_label, confidence_score)
    """
    model.eval()

    # Tokenize
    inputs = tokenizer(
        str(headline),
        str(article_body),
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    # Generate global attention mask
    input_ids = inputs["input_ids"][0].tolist()
    global_attn = [0] * len(input_ids)

    if GLOBAL_ATTENTION_MODE == "cls_only":
        global_attn[0] = 1
    elif GLOBAL_ATTENTION_MODE == "headline_and_cls":
        global_attn[0] = 1
        for j, token_id in enumerate(input_ids):
            if j == 0:
                continue
            if token_id == tokenizer.sep_token_id:
                break
            global_attn[j] = 1

    inputs["global_attention_mask"] = torch.tensor([global_attn])

    # Move to device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Predict
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        pred_id = torch.argmax(probs, dim=-1).item()
        confidence = probs[0][pred_id].item()

    predicted_label = id2label[pred_id]
    return predicted_label, confidence


# Example usage
example_headline = "NASA Confirms Evidence of Water on Mars"
example_body = (
    "Scientists at NASA have announced new findings that suggest the presence "
    "of liquid water on the surface of Mars. The discovery was made using data "
    "from the Mars Reconnaissance Orbiter, which detected hydrated salts on "
    "slopes where mysterious streaks are seen."
)

label, confidence = predict_stance(example_headline, example_body)
print(f"Headline:   {example_headline}")
print(f"Predicted:  {label}")
print(f"Confidence: {confidence:.4f}")

# Final Summary

## Model Configuration

| Parameter | Value |
|---|---|
| Model | allenai/longformer-base-4096 |
| Task | Multi-Class Classification (4 classes) |
| Labels | agree, disagree, discuss, unrelated |
| MAX_LENGTH | 2048 (adjustable based on EDA) |
| Global Attention | headline_and_cls |

## Training Configuration

| Parameter | Value |
|---|---|
| Batch Size (per device) | 2 |
| Gradient Accumulation | 8 |
| Effective Batch Size | 16 |
| Learning Rate | 2e-5 |
| Weight Decay | 0.01 |
| Warmup Ratio | 0.1 |
| Epochs | 5 (max) |
| Early Stopping | patience=2 |
| FP16 | True |
| Best Model Metric | F1 Macro |

## Evaluation Metrics

- Accuracy
- Precision (Macro)
- Recall (Macro)
- F1 Score (Macro)
- F1 Score (Weighted)
- Classification Report (per class)
- Confusion Matrix